# DQ: POS completeness monitor
Spine (every store x day) so a fully missing day shows as 0, not as "no row". Status per store-day:
- `ok`: on-time volume >= 50% of the trailing 7-day baseline
- `late_data`: low on the day, recovered once late files arrived (pipeline issue)
- `expected_holiday`: genuinely low on a known closure (Christmas, Thanksgiving)
- `low_volume_anomaly`: genuinely low, no known reason (business issue, investigate)

In [0]:
%run ./_common

In [0]:
NB = "23_dq_completeness"
spark.sql(f"""CREATE TABLE IF NOT EXISTS {CAT}.silver.calendar AS
  SELECT to_date(date) AS date, weekday, event_name_1, event_type_1, event_name_2, event_type_2
  FROM read_files('/Volumes/{CAT}/bronze/reference/calendar.csv', format => 'csv', header => true)""")

spark.sql(f"""
CREATE OR REPLACE TABLE {CAT}.ops.pos_completeness AS
WITH dates  AS (SELECT DISTINCT event_date FROM {CAT}.silver.pos_events),
     stores AS (SELECT DISTINCT store_id   FROM {CAT}.silver.pos_events),
     daily  AS (SELECT store_id, event_date, COUNT(*) AS final_events,
                       COUNT_IF(arrival_date <= event_date) AS ontime_events
                FROM {CAT}.silver.pos_events GROUP BY ALL),
     spine  AS (SELECT s.store_id, d.event_date,
                       COALESCE(x.final_events, 0)  AS final_events,
                       COALESCE(x.ontime_events, 0) AS ontime_events
                FROM stores s CROSS JOIN dates d
                LEFT JOIN daily x USING (store_id, event_date)),
     scored AS (SELECT *, AVG(final_events) OVER (PARTITION BY store_id ORDER BY event_date
                                                  ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING) AS baseline_7d
                FROM spine)
SELECT sc.*, c.event_name_1 AS calendar_event,
       ROUND(sc.ontime_events / sc.baseline_7d, 3) AS ontime_ratio,
       CASE WHEN sc.ontime_events >= 0.5 * sc.baseline_7d THEN 'ok'
            WHEN sc.final_events  >= 0.5 * sc.baseline_7d THEN 'late_data'
            WHEN c.event_name_1 IN ('Christmas','Thanksgiving')
              OR c.event_name_2 IN ('Christmas','Thanksgiving') THEN 'expected_holiday'
            ELSE 'low_volume_anomaly' END AS status
FROM scored sc
LEFT JOIN {CAT}.silver.calendar c ON c.date = sc.event_date
WHERE sc.baseline_7d IS NOT NULL""")

display(spark.sql(f"SELECT status, COUNT(*) AS store_days FROM {CAT}.ops.pos_completeness GROUP BY 1 ORDER BY 1"))

In [0]:
# informational: these describe the data, they do not fail the pipeline
for status in ["late_data", "expected_holiday", "low_volume_anomaly"]:
    n = spark.table(f"{CAT}.ops.pos_completeness").where(F.col("status") == status).count()
    dq_check(f"completeness_{status}_store_days", n, True, "informational", hard=False)
rows = table_count(f"{CAT}.ops.pos_completeness")
dq_check("completeness_table_not_empty", rows, rows > 0, "> 0")
dq_finish(NB)